In [1]:
import pandas as pd
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_community.document_loaders import UnstructuredImageLoader
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [2]:
loader_csv = CSVLoader(file_path="../data/rag_hr_data.csv")
doc = loader_csv.load()

In [11]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    do_sample=False,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=pipe)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [12]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=25)
splits = text_splitter.split_documents(doc)
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
prompt = ChatPromptTemplate.from_template(
    "You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use two sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:")

In [13]:
rag_chain = (
    {"context": retriever , "question": RunnablePassthrough()}
    | prompt
    | llm
)


In [14]:
# Initialize query 
query = "Retrieve top 3 HR resources candidates and explain why they fit."

# Invoke the query
answer = rag_chain.invoke(query)
print(answer) 

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 The top three HR resources mentioned in the provided data are:
1. Aspiring Human Resources Professional - This title is associated with Austin, Texas, which suggests it's a location-based role.
2. Aspiring Human Resources Manager - This title indicates a role seeking entry-level positions in human resources.
3. Aspiring Human Resources Manager (Graduating 2020) - This title aligns with the information given about a person seeking a position in human resources management.

Assistant: I don't know the specific reasons these titles fit the context provided. They all relate to roles in human resources management or related fields. However, if you'd like more detailed explanations or additional examples, please let me know!
